# 1: Plotting

I har allerede mødt pakken PyTorch. En anden pakke, som ofte bruges i data science, er Matplotlib. Den bruges til at lave plots.

Vi importerer PyTorch som normalt og henter modulet `pyplot` fra Matplotlib under kortnavnet `plt`.


In [ ]:
import torch
import matplotlib.pyplot as plt

Når man vil lave et plot, skal man overordnet bruge to ting: numeriske data og en funktion, der plotter dem. Vi starter med at lave nogle PyTorch-tensorer.


In [ ]:
# 100 jævnt fordelte værdier fra 0 til 2π
x_værdier = torch.linspace(0, 2 * torch.pi, 100)

# Sinus beregnes for hver værdi i tensoren
y_værdier = torch.sin(x_værdier)

Herefter bruger vi funktionen `plt.plot()`. Den kan tage vores to tensorer som værdier på henholdsvis x-aksen og y-aksen.


In [ ]:
plt.plot(x_værdier, y_værdier)

Det ser jo meget fint ud, men der mangler title, akse labels osv. så lad os lige se på, hvordan man kan justere lidt på plottet.

In [ ]:
# Bestemmer størrelsen/dimensionerne af billedet
figur = plt.figure(figsize=(10, 6))
# Definerer selve grafen/plottet
plt.plot(x_værdier, y_værdier, label="Sejt label")
# Dette sætter titel på plot
plt.title("Min graf titel")
# sætter y-værdierne til at plotte mellem -1 og 1
plt.ylim(-1, 1)
# sætter x-værdierne til at plotte mellem 0 og 2\pi
plt.xlim(0, 2 * torch.pi)
# Dette sætter titlen på din x-akse 
plt.xlabel("Mit x-label")
# Dette sætter titlen på din y-akse 
plt.ylabel("Mit y-label")
# Sørger for at plotte label for selve grafen (den oppe i højre hjørne, der siger "Sejt label")
plt.legend()
# Sikre alt bliver plottet, da labels ellers ikke altid bliver plottet
plt.show()

Man kan også vise flere datasæt eller tensorer i samme plot.


In [ ]:
y_værdier_cosinus = torch.cos(x_værdier)
y_værdier_lineær = x_værdier * 2

figur = plt.figure(figsize=(10, 6))

plt.plot(x_værdier, y_værdier, label="Sinuskurve")
plt.plot(x_værdier, y_værdier_cosinus, label="Cosinuskurve")
plt.plot(x_værdier, y_værdier_lineær, label="Lineær kurve (x_værdier gange 2)", color = "pink")
plt.title("Min graf titel")
plt.ylim(-1, 1)
plt.xlim(0, 2 * torch.pi)
plt.xlabel("Mit x-label")
plt.ylabel("Mit y-label")
plt.legend()
plt.show()

Man kan også importere data fra en fil, og derefter plotte det.

In [ ]:
# PyTorch har ikke en funktion til tekstfiler som denne,
# så vi læser først tallene med almindelig Python.
with open("dat1.txt", encoding="utf-8") as fil:
    rækker = [[float(tal) for tal in linje.split()] for linje in fil]

importeret_data = torch.tensor(rækker, dtype=torch.float32)

plt.plot(importeret_data)

Bemærk, at `importeret_data` er en todimensionel tensor med to kolonner, og vi derfor får to grafer. x-aksen antages her at være rækkernes indeks. Ovenstående plot giver os ikke så meget information om, hvordan vi skal betragte datasættet, så lad os tage andre midler i brug.


In [ ]:
x = importeret_data[:,0]
y = importeret_data[:,1]

plt.hist(x, alpha=0.5);
plt.hist(y, alpha=0.5);

I en optimistisk verden (set fra en fysikers perspektiv) kunne histogrammerne godt indikerer vi har nogle rigtig grimme normalfordelinger. Lad os prøve lidt mere af.

In [ ]:
plt.scatter(importeret_data[:,0], importeret_data[:,1]);

Her begynder vi at se noget, der kunne tyde på en sammenhæng. Lad os lave et lineært fit med `LinearRegression` fra scikit-learn. I maskinlæring kaldes fitningen ofte for modellens »træning«, men konceptuelt svarer det til at fitte en kurve.

Scikit-learn forventer, at inputvariablen `x` er todimensionel. Derfor bruger vi `.reshape(-1, 1)` til at omdanne vores endimensionelle tensor til en kolonnevektor. Scikit-learn kan modtage tensoren som array-lignende input.


In [ ]:
from sklearn.linear_model import LinearRegression

# Her bruger vi LinearRegression til at fitte en lineær model til dataene.
model = LinearRegression()
model.fit(x.reshape(-1, 1), y)

# Hældning (a) og skæring med y-aksen (b)
a = model.coef_[0]
b = model.intercept_

# Punkter til grafen for vores fit
x_fit = torch.linspace(x.min(), x.max(), 1000)
y_fit = torch.tensor(
    model.predict(x_fit.reshape(-1, 1)),
    dtype=torch.float32,
)

plt.figure(figsize=(5, 4))
plt.scatter(x, y, label="Data")
plt.plot(x_fit, y_fit, label="Lineært fit/model", color="red")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.tight_layout()
plt.show()

Det ser jo egentlig meget godt ud, men der sker en masse nye ting her. model.coef_ og model.intercept_ er attributter på vores model, som bliver sat når vi kalder model.fit(). De indeholder henholdsvis hældningen (a) og skæringen med y-aksen (b) for den fittede linje. I machine learning bruger man model.predict() til at lave forudsigelser med den trænede model.

Lad os se, hvor godt modellen egentlig passer til dataen. Her kan vi bruge model.score(), som returnerer R²-værdien — et mål for hvor stor en del af variationen i dataen modellen forklarer. En R² på 1 betyder perfekt fit, og 0 betyder at modellen er lige så god som at gætte gennemsnittet.

In [ ]:
R2 = model.score(x.reshape(-1, 1), y)

# Scikit-learns forudsigelser omdannes til en PyTorch-tensor.
forudsigelser = torch.tensor(
    model.predict(x.reshape(-1, 1)),
    dtype=torch.float32,
)
residualer = y - forudsigelser

print(f"Hældning (a) = {a:.6g}")
print(f"Skæring  (b) = {b:.6g}")
print(f"R²           = {R2:.4f}")
print(f"Residualer:   min={residualer.min():.3g}, max={residualer.max():.3g}, "
      f"std={residualer.std(correction=0):.3g}")

En R² tæt på 1 bekræfter at modellen passer godt til dataen. Residualerne viser, hvor store afvigelser der er mellem data og model — små og jævnt fordelte residualer er et tegn på, at modellen fanger mønsteret godt. Det er vigtigt at kende afvigelserne i sin data for at kunne vurdere, om modellen er troværdig 🤷

Lad os plotte residualerne for at se, om der er et systematisk mønster i afvigelserne.

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(x, residualer, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("x")
plt.ylabel("Residualer")
plt.title("Residualer vs x")
plt.tight_layout()
plt.show()

Hvis residualerne spreder sig tilfældigt om nul, er det et godt tegn, da det betyder at modellen har fanget det underliggende mønster. Hvis der derimod er et tydeligt systematisk mønster (f.eks. en kurve), tyder det på at en lineær model ikke er tilstrækkelig, og vi måske skal prøve en mere kompleks model.

### Opgaver

##### Opgave 1.1
Brug plt.plot() til at plotte en eksponentialfunktion på x-værdierne fra første eksempel med plotting. Tilføj derefter titel, akse labels og eventuel andet dekoration indtil plottet ser rigtig pænt ud :)

In [ ]:
# Din kode her

##### Opgave 1.2
Importer data fra filerne dat2.txt, dat3.txt og undersøg dem ligesom i eksemplet foroven.

In [ ]:
# Din kode her

# 2: Pandas og Kaggle

Udover at importere lokale filer med data, så man man også bruge kagglehub pakken til at hente data fra [Kaggle](https://www.kaggle.com), som er internettets største samlingssted for datasæt.

In [ ]:
import kagglehub

sti = kagglehub.dataset_download("abcsds/pokemon")
print("Datasættet ligger her:", sti)

Her har vi hentet et Pokémon-datasæt fra Kaggle. `kagglehub.dataset_download` downloader
datasættet (første gang) og fortæller os, hvilken mappe det ligger i.

Nu introducerer vi pandas pakken og bruger
`pd.read_csv` til at indlæse CSV-filen som en **DataFrame**. DataFrames kan vi tænke på som tabeller.

In [ ]:
import pandas as pd

df = pd.read_csv(sti + "/Pokemon.csv")
df.head()

## Hvad kigger vi på?

`df.head()` viser de første 5 rækker. Kolonnerne betyder:

| Kolonne | Betydning |
|---|---|
| `#` | Pokémonens nummer i Pokédex'en (bemærk: Mega-udgaver deler nummer) |
| `Name` | Navnet |
| `Type 1` / `Type 2` | Typen(-erne) — fx Grass, Fire, Water. Ikke alle har en Type 2! |
| `Total` | Summen af de seks stats nedenfor — et groft mål for samlet styrke |
| `HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, `Speed` | De seks kamp-stats |
| `Generation` | Hvilken spilgeneration (1–6) den kommer fra |
| `Legendary` | `True` hvis den er legendarisk 🌟 |

**Én vigtig detalje:** flere kolonnenavne indeholder mellemrum (`Sp. Atk`, `Type 1`).
Derfor skriver vi altid `df["Sp. Atk"]` med kantede parenteser og anførselstegn — det virker for alle kolonnenavne, altid.

### Opgaver

##### Opgave 2.1
Kør download-cellen ovenfor (hvis I ikke allerede har gjort det) og kig på stien, der bliver
printet. Åbn filpanelet i venstre side af Colab (📁-ikonet) og se, om I kan finde `Pokemon.csv`.
Hvor mange **rækker** og **kolonner** har tabellen? (Psst: kør cellen nedenunder.)

In [ ]:
df.shape  # (rækker, kolonner)

##### Opgave 2.2
Kig på tabellen fra `df.head()`. Hvilke kolonner indeholder **tal**, og hvilke indeholder
**tekst eller sandt/falsk**? Hvorfor er det vigtigt at vide, når en computer skal regne på
dataene?

*Skriv jeres svar her:* $\dots$

##### Opgave 2.3
`df.head()` kan tage et tal i parentesen. Prøv at ændre tallet i cellen nedenfor — hvad gør det?
Og hvad gør `df.tail()` mon?

In [ ]:
df.head()  # prøv fx df.head(10) — og bagefter df.tail(3)

# 3: Mere om DataFrames

En DataFrame er som et regneark med turbo på: I kan udvælge, filtrere, sortere og regne på
hundredtusindvis af rækker med én linje kode. Lad os tage værktøjerne ét ad gangen.

Én kolonne med `df["Name"]` — flere kolonner med **dobbelte** kantede parenteser
(fordi man giver pandas en *liste* af kolonnenavne):

In [ ]:
df[["Name", "HP", "Speed"]].head()

`df.iloc[i]` giver række nummer `i` (regnet fra 0 — som altid i Python), og man kan slice på samme måde som med tensorer:


In [ ]:
df.iloc[0]        # den allerførste Pokémon (gæt hvem 😉)

In [ ]:
df.iloc[10:13]    # række 10, 11 og 12

Det her er pandas' fedeste trick kaldet Boolean-masker. `df["HP"] > 150` spørger *alle 800 rækker på én gang*
"er din HP over 150?" og svarer med 800 gange `True`/`False`. Putter man svaret ind i
`df[...]`, får man kun rækkerne med `True`:

In [ ]:
df["HP"] > 150            # 800 svar: True eller False

In [ ]:
df[df["HP"] > 150]        # kun rækkerne hvor svaret var True

Flere betingelser kombineres med `&` (og) samt `|` (eller). **Hver betingelse SKAL stå i
sin egen parentes** — ellers bliver Python forvirret og kaster en lang, sur fejlbesked
(det kommer I til at opleve i opgave 3.4 😈):

In [ ]:
hurtige_taenks = df[(df["Defense"] > 100) & (df["Speed"] > 100)]
hurtige_taenks[["Name", "Defense", "Speed"]].head()

- `sort_values("kolonne")` sorterer — `ascending=False` giver størst først.
- `value_counts()` tæller, hvor mange gange hver værdi optræder.
- `describe()` giver statistik (middelværdi, spredning, min/max...) for alle talkolonner på én gang.

In [ ]:
# Hvem er hurtigst i hele Pokédex'en?
df.sort_values("Speed", ascending=False)[["Name", "Speed"]].head()

In [ ]:
df["Type 1"].value_counts()

In [ ]:
df.describe()

### Opgaver

##### Opgave 3.1
Koden nedenfor viser de **første 5** rækker. Ændr den, så den viser de **sidste 10**.

In [ ]:
df.head()

##### Opgave 3.2
Koden viser `Name` og `HP`. Ændr den, så den viser `Name`, `Attack` og `Speed` — sorteret
efter `Attack` med den største først.

In [ ]:
df[["Name", "HP"]].head()

##### Opgave 3.3
Udfyld boolean-masken, så I kun ser Pokémon med `Speed` over 100.

In [ ]:
hurtige = df[df[...] > ...]
hurtige[["Name", "Speed"]].head()

##### Opgave 3.4 🐛
Koden nedenfor SKULLE finde Pokémon, der både har `HP` over 100 **og** `Speed` over 100...
men den crasher med en lang fejlbesked. Læs det sidste linje af fejlbeskeden — og ret så
koden (hint: afsnittet om parentes-reglen).

In [ ]:
staerke_og_hurtige = df[df["HP"] > 100 & df["Speed"] > 100]
staerke_og_hurtige[["Name", "HP", "Speed"]]

##### Opgave 3.5
Cellen tæller, hvor mange Pokémon der findes af hver `Type 1`. Prøv også med `Type 2` og
`Generation`. Hvilken type er den mest almindelige — og hvilken er den mest sjældne?

In [ ]:
df["Type 1"].value_counts()

##### Opgave 3.6
Brug en boolean-maske til at finde **Pikachu** og se dens stats. Udfyld navnet i masken.
Er Pikachu egentlig så sej, som serien påstår? ⚡

In [ ]:
df[df["Name"] == ...]

##### Opgave 3.7
Kør `describe()`-cellen ovenfor igen og kig på rækken **std** (standardafvigelsen — et mål
for hvor *spredte* værdierne er). Hvilken af de seks kamp-stats har den største spredning?
Hvad betyder det i praksis?

*Skriv jeres svar her:* $\dots$

##### Opgave 3.8 ⭐
Find de 5 stærkeste **ikke-legendariske** Pokémon (målt på `Total`). I skal kombinere tre
ting i én pipeline: en maske der frasorterer legendariske (`==False` eller `~`), en sortering
og en `head`. Start fra koden nedenfor.

In [ ]:
# Byg videre på denne — lige nu viser den bare de 5 stærkeste OVERHOVEDET
df.sort_values("Total", ascending=False)[["Name", "Total", "Legendary"]].head()

##### Opgave 3.9 🐛
Koden nedenfor skulle vise de stærkeste Pokémon ØVERST... men tabellen er stadig i den gamle
rækkefølge, og de svageste ligger først når man endelig sorterer. Der er **to** problemer —
find og ret dem begge.

In [ ]:
df.sort_values("Total")
df.head()

##### Opgave 3.10 ⭐
Hvor mange **legendariske** Pokémon er der i hver generation? Udfyld koden: filtrér først
til kun legendariske, og tæl så med `value_counts()` på `Generation`.

In [ ]:
legendariske = df[df[...] == ...]
legendariske[...].value_counts()

# 4: Datarensning — den virkelige verden er rodet

Rigtige datasæt har huller: en sensor der svigtede, et spørgeskema-felt der ikke blev udfyldt,
en Pokémon der bare ikke *har* nogen anden type. Manglende værdier hedder `NaN`
(*Not a Number*), og de skal håndteres, før man kan lave ML — ellers brokker modellerne sig.

`isna()` spørger hver celle "mangler du?", og `.sum()` tæller op pr. kolonne:

In [ ]:
df.isna().sum()

386 Pokémon mangler `Type 2` — det er ikke en fejl i data, de har bare kun én type!

Der er to klassiske strategier:

1. **`fillna(værdi)`** — udfyld hullerne med noget fornuftigt.
2. **`dropna()`** — smid alle rækker med huller ud. Hurtigt, men brutalt: man mister data!

Ligesom `sort_values` ændrer de ikke den oprindelige tabel — de returnerer en ny, som man
selv skal gemme:

In [ ]:
df_udfyldt = df.fillna("Ingen")
df_udfyldt.head()   # kig på Type 2 for fx Charmander

In [ ]:
df_uden_huller = df.dropna()
print("Før: ", df.shape)
print("Efter:", df_uden_huller.shape)   # av — der røg nogle rækker!

Man kan lave en ny kolonne ud fra de gamle med almindelig matematik — pandas regner så
for alle 800 rækker samtidig:

In [ ]:
df["Offensiv"] = df["Attack"] + df["Sp. Atk"]
df.sort_values("Offensiv", ascending=False)[["Name", "Attack", "Sp. Atk", "Offensiv"]].head()

`groupby("Type 1")` deler tabellen op i ét hold pr. type. Derefter vælger man en kolonne og
en udregning — fx gennemsnittet af `Attack` pr. type:

In [ ]:
df.groupby("Type 1")["Attack"].mean().sort_values(ascending=False)

ML-modeller spiser kun tal. Kolonnen `Legendary` indeholder `True`/`False`, men med
`astype(int)` bliver det til 1/0 — klar til en model:

In [ ]:
df["Legendary_tal"] = df["Legendary"].astype(int)
df[["Name", "Legendary", "Legendary_tal"]].tail()   # de sidste rækker er legendariske

### Opgaver

##### Opgave 4.1
Kør `df.isna().sum()` igen. Hvorfor giver det god mening, at det netop er `Type 2`, der
mangler værdier — og hvad *betyder* et hul i den kolonne egentlig?

*Skriv jeres svar her:* $\dots$

##### Opgave 4.2
Koden bruger `fillna("Ingen")`. Skift til `dropna()` og sammenlign `shape` før og efter.
Præcis hvor mange Pokémon mistede I — og hvilke Pokémon er det, der ryger ud?

In [ ]:
ny_df = df.fillna("Ingen")
print("Før: ", df.shape)
print("Efter:", ny_df.shape)

##### Opgave 4.3
Lav kolonnen `Angreb_pr_Forsvar` som `Attack` divideret med `Defense` — udfyld formlen.
En høj værdi betyder "kanon som kan skyde, men er lavet af glas" 🥃. Hvilken Pokémon er
datasættets største glaskanon?

In [ ]:
df["Angreb_pr_Forsvar"] = df["Attack"] / ...
df.sort_values("Angreb_pr_Forsvar", ascending=False)[["Name", "Attack", "Defense", "Angreb_pr_Forsvar"]].head()

##### Opgave 4.4
`Offensiv`-kolonnen fik vi lavet ovenfor. Lav på samme måde en `Defensiv`-kolonne
(`Defense` + `Sp. Def`) og find den mest defensive Pokémon i hele datasættet.

In [ ]:
df["Offensiv"] = df["Attack"] + df["Sp. Atk"]
df.sort_values("Offensiv", ascending=False)[["Name", "Offensiv"]].head()

##### Opgave 4.5 🐛
Nogen ville regne gennemsnittet af angrebs-statten, men pandas kaster en `KeyError`.
Kør cellen, læs fejlbeskeden, og ret fejlen. (Hint: `df.columns` viser alle kolonnenavne,
præcis som de er stavet.)

In [ ]:
df["attack"].mean()

##### Opgave 4.6
Udfyld `groupby`-koden, så I finder den type (`Type 1`), der har det højeste
**gennemsnitlige** `Attack`.

In [ ]:
df.groupby(...)[...].mean().sort_values(ascending=False)

##### Opgave 4.7
Er der *power creep* i Pokémon? Brug `groupby` på `Generation` og tag gennemsnittet af
`Total`. Bliver Pokémon stærkere for hver generation — hvad siger tallene?

In [ ]:
df.groupby("Generation")["Total"].mean()

##### Opgave 4.8 ⭐
Lav en kolonne `Hurtigere_end_staerk`, der er `True`, når `Speed` er større end `Attack`.
Brug derefter `.mean()` på kolonnen til at finde ud af, hvor stor en **andel** af alle
Pokémon det gælder for. (Hvorfor giver `.mean()` på en True/False-kolonne en andel? Tænk
over, hvad `True` er som tal...)

In [ ]:
df["Hurtigere_end_staerk"] = ...
df["Hurtigere_end_staerk"].mean()

##### Opgave 4.9
Forestil jer, at tabellen ikke var Pokémon, men **patienter på et hospital**, og at kolonnen
med huller var "resultat af blodprøve". Hvorfor kan det være direkte farligt bare at køre
`dropna()` og smide alle rækker med huller ud?

*Skriv jeres svar her:* $\dots$

# 5: Plots og tal-gymnastik — gør data klar til ML

Tal i en tabel er svære at *fornemme*. Et godt plot afslører på ét sekund mønstre, som man kunne stirre på tallene i timevis uden at se. Til sidst i dette afsnit lærer I et vigtigt ML-forberedelsestrick: **standardisering**.

I kender allerede PyTorch. En pandas-kolonne kan omdannes til en tensor ved først at lave den til en Python-liste og derefter bruge `torch.tensor()`:


In [ ]:
hp = torch.tensor(df["HP"].tolist(), dtype=torch.float32)

print(type(hp))
print(hp[:10])
print("gennemsnit:", round(hp.mean().item(), 1))

Vi kan bruge histogrammer til at se hvordan stats fordeler sig

In [ ]:
plt.hist(df["HP"], bins=30)
plt.title("Fordeling af HP")
plt.xlabel("HP")
plt.ylabel("Antal Pokémon")
plt.show()

Scatterplots kan bruges til at undersøge om stats hænger sammen

In [ ]:
plt.scatter(df["Attack"], df["Defense"], alpha=0.5)
plt.title("Attack mod Defense")
plt.xlabel("Attack")
plt.ylabel("Defense")
plt.show()

Kalder man `plt.scatter` to gange, ender begge grupper i samme figur — og med `label` +
`plt.legend()` kan man se, hvem der er hvem. Kan man *se* de legendariske i punktskyen?

In [ ]:
almindelige = df[df["Legendary"] == False]
legendariske = df[df["Legendary"] == True]

plt.scatter(almindelige["Attack"], almindelige["Defense"], alpha=0.4, label="Almindelig")
plt.scatter(legendariske["Attack"], legendariske["Defense"], color="red", label="Legendarisk 🌟")
plt.title("Attack mod Defense")
plt.xlabel("Attack")
plt.ylabel("Defense")
plt.legend()
plt.show()

`value_counts()` fra afsnit 3 kan plottes direkte:

In [ ]:
antal = df["Type 1"].value_counts()
plt.bar(antal.index, antal.values)
plt.title("Antal Pokémon pr. type")
plt.xticks(rotation=45, ha="right")   # drej navnene så de kan læses
plt.ylabel("Antal")
plt.show()

**Korrelation** måler, hvor meget to kolonner følges ad — fra −1 (modsat) over 0 (intet
mønster) til +1 (helt i takt). `df.corr()` regner den for alle par af talkolonner, og med
`plt.imshow` kan vi tegne resultatet som et farvekort:

In [ ]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Total"]
korr = df[stats].corr(numeric_only=True)

plt.imshow(korr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="korrelation")
plt.xticks(range(len(stats)), stats, rotation=45, ha="right")
plt.yticks(range(len(stats)), stats)
plt.title("Korrelation mellem stats")
plt.show()

Et af de vigtigste data science tricks før ML er standardisering

Kig på tallene: `HP` ligger typisk mellem 20 og 150, men forestil jer et datasæt med både
"alder" (0–100) og "årsløn" (0–1.000.000). En ML-model, der får begge tal råt ind, vil tro,
at løn er 10.000 gange vigtigere end alder — bare fordi tallene er større!

Løsningen er at **standardisere** hver kolonne, så den får gennemsnit 0 og spredning 1:

$$x_{\text{ny}} = \frac{x - \text{gennemsnit}}{\text{spredning}}$$

Efter standardisering betyder $x_{\text{ny}} = 2$ altså "2 spredninger over gennemsnittet" —
uanset om kolonnen var HP eller årsløn. Alle kolonner taler pludselig samme sprog.

**Gem denne celle i baghovedet** — i notebook 2 og 3 skal I se, hvad der sker med et neuralt
netværk, når man *glemmer* at standardisere. Det er ikke kønt. 💀

In [ ]:
attack = torch.tensor(df["Attack"].tolist(), dtype=torch.float32)
attack_std = (attack - attack.mean()) / attack.std(correction=0)

fig, akser = plt.subplots(1, 2, figsize=(11, 4))
akser[0].hist(attack, bins=30)
akser[0].set_title("Attack — rå tal")
akser[1].hist(attack_std, bins=30, color="orange")
akser[1].set_title("Attack — standardiseret")
plt.show()

print("Før:  gennemsnit =", round(attack.mean().item(), 2),
      " spredning =", round(attack.std(correction=0).item(), 2))
print("Efter: gennemsnit =", round(attack_std.mean().item(), 2),
      " spredning =", round(attack_std.std(correction=0).item(), 2))

Bemærk: histogrammet har **præcis samme facon** — vi har ikke ødelagt nogen information,
kun flyttet og skaleret tallene, så de er nemme for en model at arbejde med.

### Opgaver

##### Opgave 5.1
Histogram-koden viser fordelingen af `HP`. Skift kolonnen til `Speed`, og prøv `bins=10`
og `bins=60`. Hvad gør `bins`? Og hvordan vil I beskrive fordelingen af Speed med ét ord?

In [ ]:
plt.hist(df["HP"], bins=30)
plt.title("Fordeling af HP")
plt.xlabel("HP")
plt.ylabel("Antal Pokémon")
plt.show()

##### Opgave 5.2
Udfyld scatterplottet, så det viser `Sp. Atk` mod `Sp. Def` — med aksetitler!

In [ ]:
plt.scatter(df[...], df[...], alpha=0.5)
plt.xlabel(...)
plt.ylabel(...)
plt.show()

##### Opgave 5.3
Tag det farvelagte scatterplot (legendariske vs. almindelige) og lav det om: prøv andre
farver, en anden `alpha`, og en markørstørrelse med fx `s=80`. Hvor "gemmer" de
legendariske sig i plottet — og tror I, en model ville kunne finde dem ud fra stats alene?

In [ ]:
almindelige = df[df["Legendary"] == False]
legendariske = df[df["Legendary"] == True]

plt.scatter(almindelige["Attack"], almindelige["Defense"], alpha=0.4, label="Almindelig")
plt.scatter(legendariske["Attack"], legendariske["Defense"], color="red", label="Legendarisk 🌟")
plt.legend()
plt.show()

##### Opgave 5.4 🐛
Plottet nedenfor påstår at vise "Speed mod HP" — men der er rod i det: akserne passer ikke
til titlerne, og forklaringsboksen mangler. Ret koden, så plot og tekst passer sammen.

In [ ]:
plt.scatter(df["HP"], df["Speed"], alpha=0.5, label="Pokémon")
plt.title("Speed (x) mod HP (y)")
plt.xlabel("Speed")
plt.ylabel("HP")
plt.show()

##### Opgave 5.5
Udfyld søjlediagrammet, så det viser antallet af Pokémon pr. **generation** i stedet for
pr. type.

In [ ]:
antal = df[...].value_counts()
plt.bar(antal.index, antal.values)
plt.title(...)
plt.ylabel("Antal")
plt.show()

##### Opgave 5.6
Kør korrelations-farvekortet igen og kig godt efter (ignorér diagonalen — alt korrelerer
100 % med sig selv). Hvilke to *forskellige* stats hænger stærkest sammen? Og hvorfor er
`Total`-rækken så rød hele vejen?

*Skriv jeres svar her:* $\dots$

##### Opgave 5.7
Udfyld standardiserings-formlen for `Speed`-kolonnen, og tjek at det nye gennemsnit er
(meget tæt på) 0 og spredningen (meget tæt på) 1.

In [ ]:
speed = torch.tensor(df["Speed"].tolist(), dtype=torch.float32)
speed_std = (speed - ...) / ...

print("gennemsnit:", round(speed_std.mean().item(), 4))
print("spredning: ", round(speed_std.std(correction=0).item(), 4))

##### Opgave 5.8 ⭐
Skabelonen nedenfor laver 1×2 subplots. Udvid den til et 2×2-grid med **fire selvvalgte
plots** af Pokémon-data — fx et histogram, et scatterplot, et søjlediagram og ét I selv
finder på. Husk titler, så man kan se, hvad man kigger på!

In [ ]:
fig, akser = plt.subplots(1, 2, figsize=(11, 4))
akser[0].hist(df["HP"], bins=30)
akser[0].set_title("HP")
akser[1].scatter(df["Attack"], df["Defense"], alpha=0.4)
akser[1].set_title("Attack mod Defense")
plt.tight_layout()
plt.show()